In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/src/annotated/checkpoints/post_cell_23.pickle

trying: ['dirname']
me:  0
trying: ['np']
me:  0
trying: ['overall_mean']
me:  31
trying: ['filenames']
me:  0
trying: ['mismatch_names']
me:  15
trying: ['popul_df']
me:  23
trying: ['sns']
me:  0
trying: ['vlgr_df']
me:  23
trying: ['combined_df']


me:  47
trying: ['filename']
me:  0
trying: ['orig_output']
me:  48
trying: ['personality_ranking']
me:  45
trying: ['factor']
me:  3
trying: ['species_ranking']
me:  41
trying: ['pd']
me:  0
trying: ['plt']
me:  0
Declaring variable dirname
Declaring variable np
Declaring variable filenames
Declaring variable sns
Declaring variable filename
Declaring variable pd
Declaring variable plt
Declaring variable factor
Declaring variable mismatch_names
Declaring variable popul_df
Declaring variable vlgr_df
Declaring variable overall_mean
Declaring variable species_ranking
Declaring variable personality_ranking
Declaring variable combined_df
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 24 ###
# Melt the two style columns into a long format so we only need one groupby
_melted = combined_df.melt(
    id_vars='overall_ranking',
    value_vars=['Style 1', 'Style 2'],
    var_name='style_type',
    value_name='style'
)

# Compute the mean overall ranking per style (across both style types) in one GPU groupby
_ranked = _melted.groupby(['style_type', 'style'], as_index=False).mean()

# Split out Style 1, sort and rename
style_ranking1 = (
    _ranked[_ranked['style_type'] == 'Style 1']
    .sort_values('overall_ranking')
    .rename(columns={'style': 'Style 1'})
    [['Style 1', 'overall_ranking']]
)

# For Style 2 we need to preserve the original reset_index–based index from the per-style groupby
# 1) Build a mapping from each Style 2 label to its original reset_index position
_style2_init = combined_df.groupby('Style 2').mean(numeric_only=True)[['overall_ranking']].reset_index()
_style2_init['orig_idx'] = _style2_init.index
_style2_map = _style2_init[['Style 2', 'orig_idx']]

# 2) Compute the sorted Style 2 ranking
style_ranking2 = (
    _ranked[_ranked['style_type'] == 'Style 2']
    .sort_values('overall_ranking')
    .rename(columns={'style': 'Style 2'})
    [['Style 2', 'overall_ranking']]
)

# 3) Merge in the original index positions and set them as the DataFrame index
style_ranking2 = style_ranking2.merge(_style2_map, on='Style 2', how='left')
style_ranking2 = style_ranking2.set_index('orig_idx', drop=True)[['Style 2', 'overall_ranking']]


CPU times: user 91.8 ms, sys: 43.9 ms, total: 136 ms
Wall time: 136 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/src/rewritten/o4_mini_high/checkpoints/post_cell_24_try_3.pickle

migration speed (bps): 805430481.139848
---------------------------
variables to migrate:
np 72
pd 72
sns 72
vlgr_df 48
plt 72
personality_ranking 48
style_ranking2 48
mismatch_names 48
species_ranking 48
factor 28
dirname 187
overall_mean 32
orig_output 48
filenames 88
combined_df 48
popul_df 48
filename 62
style_ranking1 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/src/rewritten/o4_mini_high/checkpoints/post_cell_24_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['popul_df', 'vlgr_df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['popul_df', 'vlgr_df']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['popul_df', 'vlgr_df']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['popul_df', 'vlgr_df']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['popul_df', 'vlgr_df']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['popul_df', 'vlgr_df']
Modified dataframes
======= Cell 6 =======
Input variables []
Active variables []
Intermediate variables []
Future var

In [7]:

with open("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/src/opt_cell_exec_info_24_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[24], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
style_ranking1_opt = style_ranking1
style_ranking2_opt = style_ranking2
%LoadCheckpoint /home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/src/annotated/checkpoints/post_cell_24.pickle
assert compare_df(style_ranking1_opt, style_ranking1)
assert compare_df(style_ranking2_opt, style_ranking2)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
